In [ ]:
# Import everything we need
import sys, requests
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path (for Jupyter notebooks)
project_root = Path.cwd().parent.parent.parent  # Go up from notebooks/ -> test/ -> src/ -> project root
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

# Import from your project
from config.constants import supabase, EMBEDDING_MODEL
from ingest.embedding_import import generate_embeddings

print("✅ Imports successful!")
print(f"Using embedding model: {EMBEDDING_MODEL}")

In [ ]:
# Cell: Inspect Tables Structure
print("="*70)
print("INSPECTING DATABASE TABLES")
print("="*70)

# 1. Check toc_nodes structure
print("\n1. TOC_NODES TABLE")
print("-"*70)
toc_sample = supabase.from_('toc_nodes').select('*').limit(3).execute()

if toc_sample.data:
    print(f"Total rows: {len(toc_sample.data)}")
    print(f"\nColumns: {list(toc_sample.data[0].keys())}")
    print(f"\nFirst 3 rows:")
    for i, row in enumerate(toc_sample.data, 1):
        print(f"\n  Row {i}:")
        for key, value in row.items():
            print(f"    {key:15s}: {value}")

# 2. Check chunks structure  
print("\n\n2. CHUNKS TABLE")
print("-"*70)
chunks_sample = supabase.from_('chunks').select('*').limit(2).execute()

if chunks_sample.data:
    print(f"Total rows: Check count separately")
    print(f"\nColumns: {list(chunks_sample.data[0].keys())}")
    print(f"\nFirst 2 rows:")
    for i, row in enumerate(chunks_sample.data, 1):
        print(f"\n  Row {i}:")
        for key, value in row.items():
            if key == 'embedding':
                print(f"    {key:15s}: vector with {len(value) if value else 0} dimensions")
            elif key == 'text':
                print(f"    {key:15s}: {value[:100]}..." if value else f"    {key:15s}: None")
            else:
                print(f"    {key:15s}: {value}")

# 3. Check the relationship
print("\n\n3. CHECKING TOC-CHUNKS RELATIONSHIP")
print("-"*70)
sample_chunk = chunks_sample.data[0]
toc_node_id = sample_chunk['toc_node_id']

print(f"Sample chunk references toc_node_id: {toc_node_id}")

# Get the corresponding toc_node
toc_node = supabase.from_('toc_nodes').select('*').eq('id', toc_node_id).execute()
if toc_node.data:
    print(f"\nCorresponding ToC node:")
    print(f"  id: {toc_node.data[0]['id']}")
    print(f"  node_id: {toc_node.data[0]['node_id']}")
    print(f"  title: {toc_node.data[0]['title']}")
    print(f"  level: {toc_node.data[0]['level']}")
    print(f"  pages: {toc_node.data[0]['page_start']}-{toc_node.data[0]['page_end']}")

# 4. Understand chunk_id format
print("\n\n4. CHUNK_ID FORMAT ANALYSIS")
print("-"*70)
for i, chunk in enumerate(chunks_sample.data[:3], 1):
    chunk_id = chunk['chunk_id']
    parts = chunk_id.split('::')
    print(f"\nChunk {i}: {chunk_id}")
    print(f"  Parts: {parts}")
    print(f"    - doc_key: {parts[0] if len(parts) > 0 else 'N/A'}")
    print(f"    - section_id: {parts[1] if len(parts) > 1 else 'N/A'}")
    print(f"    - chunk_seq: {parts[2] if len(parts) > 2 else 'N/A'}")

In [ ]:
try:
    toc_response = supabase.from_('toc_nodes').select('*', count='exact').limit(3).execute()
    print(f"✅ toc_nodes table: {toc_response.count} rows")
    if toc_response.data:
        sample = toc_response.data[0]
        print(f"   Sample: Level {sample['level']}: {sample['title'][:50]}")
except Exception as e:
    print(f"❌ toc_nodes error: {e}")

# Check chunks
try:
    chunks_response = supabase.from_('chunks').select('*', count='exact').limit(1).execute()
    print(f"✅ chunks table: {chunks_response.count} rows")
    if chunks_response.data:
        sample = chunks_response.data[0]
        print(f"   Sample chunk: {sample['chunk_id']}")
        print(f"   Section: {sample['section_title']}")
        print(f"   Has embedding: {sample.get('embedding') is not None}")
        if sample.get('embedding'):
            print(f"   Embedding dimension: {len(sample['embedding'])}")
except Exception as e:
    print(f"❌ chunks error: {e}")

print("\n" + "="*70)
print("STEP 3: LOAD EMBEDDING MODEL (reusing from ingest)")
print("="*70)

# Test embedding generation
test_text = "What a RAG?"
print(f"Test query: '{test_text}'")

embedding = generate_embeddings(
    texts=[test_text],
    model_name=EMBEDDING_MODEL,
    show_progress=False
)

print(f"✅ Generated embedding")
print(f"   Shape: {embedding.shape}")
print(f"   Dimension: {embedding.shape[1]}")

In [ ]:
# Cell 2: Check what's in the database
response = supabase.from_('toc_nodes').select('*').limit(10).execute()

print("First 10 sections from AI Engineering book:\n")
for section in response.data:
    indent = "  " * (section['level'] - 1)
    print(f"{indent}H{section['level']}: {section['title']}")
    print(f"{indent}   Pages {section['page_start']}-{section['page_end']}")

In [ ]:
# Cell 3: Simple direct query approach (no RPC function)
def search(query: str, top_k: int = 5):
    """Direct pgvector search without RPC function"""
    
    query_embedding = generate_embeddings(
        texts=[query],
        model_name=EMBEDDING_MODEL,
        show_progress=False
    )[0].tolist()
    
    # Direct SQL query using Supabase
    # Since RPC is giving issues, let's use the simpler approach
    try:
        # Try getting chunks and computing similarity in Python
        all_chunks = supabase.from_('chunks').select('*').limit(100).execute()
        
        if not all_chunks.data:
            return []
        
        # Compute similarities
        results = []
        for chunk in all_chunks.data:
            if chunk.get('embedding'):
                # Compute cosine similarity
                chunk_emb = np.array(chunk['embedding'])
                query_emb = np.array(query_embedding)
                similarity = np.dot(query_emb, chunk_emb) / (
                    np.linalg.norm(query_emb) * np.linalg.norm(chunk_emb)
                )
                
                chunk['similarity'] = float(similarity)
                results.append(chunk)
        
        # Sort by similarity and return top_k
        results.sort(key=lambda x: x['similarity'], reverse=True)
        return results[:top_k]
        
    except Exception as e:
        print(f"Error: {e}")
        return []

print("✅ search() ready (direct similarity computation)")

In [ ]:
# Cell 4: Test with AI Engineering queries
queries = [
    "What is prompt engineering?",
    "How do you fine-tune a language model?",
    "What are embeddings?",
    "Explain RAG architecture",
    "What is few-shot learning?"
]

for query in queries:
    print(f"\n{'='*70}")
    print(f"Query: {query}")
    print('='*70)
    results = search(query, top_k=3)
    
    for i, r in enumerate(results, 1):
        print(f"\n{i}. [{r.get('similarity', 0):.3f}] {r['section_title']}")
        print(f"   Pages {r['page_start']}-{r['page_end']}")
        print(f"   {r['text'][:200]}...")